<a href="https://colab.research.google.com/github/ceicyterugile/E-Commerce-Marketing-Campaigns-Session-Duration-Weekday-Trend-Analytics/blob/main/%E2%80%9EUntitled9_ipynb%E2%80%9C_kopija.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sprint: Marketing Analytics
# Project: **Marketing Campaign Comparison**
**Goal:** You have a follow-up task from your marketing manager to identify the overall trends of all marketing campaigns on your e-commerce site. She is particularly interested in finding out if users tend to spend more time on your website on certain weekdays and how that behavior differs across campaigns.
Create a presentation centered around the dynamic weekday duration, focusing on differences between marketing campaigns.
See whether you can apply 1-2 techniques learned in this or other modules throughout the course material to enhance your presentation on this subject.
Explore the data. See whether there are interesting data points that can give more insights to your presentation.
Provide analytical insights; what are the drawbacks of this analysis, and what further analysis could you recommend?

**Data:** You should use the **turing_college.raw_events** table to answer this question. Data in the table contains records from 2020-11-01 until 2021-01-31. Write a SQL query that would extract data from BigQuery, make a visualization using your preferred data visualization tool (Google Sheets / Tableau / Looker Studio), and briefly comment on your findings. As we do not have session identifiers in the dataset, you will have to come up with your own logic for how you will model sessions. Keep in mind that a single user can come to your website on multiple days, and if you were to calculate time on the website, this may have an impact on this metric.




Here is a SQL query with several steps, including defining a session, calculating session duration, and then aggregating the results by campaign and weekday.
Since there isn't session ID, a common approach is to model a session as a series of events from the same user within a certain time frame (e.g., 30 minutes).

**SQL Query for Campaign Trends and Time-on-Site by Weekday**

In [ ]:
WITH RankedEvents AS (
    -- 1. Rank events for each user chronologically
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_pseudo_id
            ORDER BY event_timestamp
        ) as rn
    FROM
        `turing_data_analytics.raw_events`
    -- Filter to include only relevant events (e.g., 'page_view', 'scroll', 'add_to_cart', etc.)
    -- and records with a campaign name
    WHERE
        campaign IS NOT NULL
        AND campaign != ''
        -- Optionally filter event_name if you only want to consider certain interaction events
        -- AND event_name IN ('page_view', 'user_engagement', 'add_to_cart', 'purchase')
),
SessionStartFlags AS (
    -- 2. Identify the start of a new session
    SELECT
        *,
        -- Calculate the time difference (in seconds) between the current and the previous event for the same user
        (event_timestamp - LAG(event_timestamp, 1, event_timestamp) OVER (
            PARTITION BY user_pseudo_id
            ORDER BY event_timestamp
        )) / 1000000 AS time_diff_seconds, -- Assuming event_timestamp is in microseconds

        -- Flag a new session if:
        -- a) It's the user's first event (rn = 1) OR
        -- b) The time difference to the previous event is greater than a session timeout (e.g., 30 minutes = 1800 seconds)
        CASE
            WHEN rn = 1 OR (event_timestamp - LAG(event_timestamp) OVER (
                PARTITION BY user_pseudo_id
                ORDER BY event_timestamp
            )) / 1000000 > 1800 -- 30 minutes in seconds
            THEN 1
            ELSE 0
        END AS is_new_session
    FROM
        RankedEvents
),
SessionIDs AS (
    -- 3. Assign a unique Session ID to each session for each user
    SELECT
        *,
        -- Cumulatively sum the 'is_new_session' flag to create a session group ID
        SUM(is_new_session) OVER (
            PARTITION BY user_pseudo_id
            ORDER BY event_timestamp
        ) AS session_group_id,

        -- Extract the day of the week and a numerical representation for ordering
        -- (Specific function depends on your SQL dialect, e.g., DAYOFWEEK, EXTRACT(DOW), DATE_PART('dow'))
        -- The example uses PostgreSQL/BigQuery style functions for demonstration
        EXTRACT(DAYOFWEEK FROM TIMESTAMP_MICROS(event_timestamp)) AS weekday_number,

        -- Convert the weekday number to a name for better reporting
        CASE EXTRACT(DAYOFWEEK FROM TIMESTAMP_MICROS(event_timestamp))
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
            WHEN 7 THEN 'Sunday'
        END AS session_weekday_name
    FROM
        SessionStartFlags
),
SessionDuration AS (
    -- 4. Calculate the duration of each session
    SELECT
        user_pseudo_id,
        campaign,
        session_group_id,
        session_weekday_name,
        weekday_number,

        -- The duration is the difference between the last event and the first event in the session
        -- Use MIN/MAX on event_timestamp within the session group
        (MAX(event_timestamp) - MIN(event_timestamp)) / 1000000 AS session_duration_seconds

    FROM
        SessionIDs
    GROUP BY
        1, 2, 3, 4, 5
)
-- 5. Final Aggregation: Find the average time on site by campaign and weekday
SELECT
    campaign,
    weekday_number,
    session_weekday_name,
    COUNT(DISTINCT user_pseudo_id) AS total_users_in_campaign_on_weekday,
    COUNT(session_group_id) AS total_sessions,
    -- Calculate the average session duration (Time on Site)
    AVG(session_duration_seconds) AS average_time_on_site_seconds,
    -- Calculate the average number of sessions per user
    CAST(COUNT(session_group_id) AS NUMERIC) / COUNT(DISTINCT user_pseudo_id) AS avg_sessions_per_user
FROM
    SessionDuration
GROUP BY
    1, 2, 3
ORDER BY
    campaign,
    weekday_number;

**SQL Query results**

In [ ]:
import pandas as pd

df = pd.read_csv('/content/bquxjob_4acc5ab4_199a4b60e09.csv')
display(df)

,campaign,weekday_number,session_weekday_name,total_users_in_campaign_on_weekday,total_sessions,average_time_on_site_seconds,avg_sessions_per_user
0,(data deleted),1,Monday,654,690,192.589708,1.055046
1,(data deleted),2,Tuesday,980,1020,164.192196,1.040816
2,(data deleted),3,Wednesday,1147,1208,173.282816,1.053182
3,(data deleted),4,Thursday,1108,1171,185.055336,1.056859
4,(data deleted),5,Friday,1057,1103,194.414707,1.043519
...,...,...,...,...,...,...,...
66,NewYear_V1,7,Sunday,2,2,5.736445,1.000000
67,NewYear_V2,4,Thursday,5,5,18.534326,1.000000
68,NewYear_V2,5,Friday,25,27,104.584791,1.080000
69,NewYear_V2,6,Saturday,4,5,20.218950,1.250000


In [ ]:
import altair as alt

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X('session_weekday_name', sort=weekday_order, title='Weekday'),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    color=alt.Color('campaign', title='Campaign'),
    tooltip=['campaign', 'session_weekday_name', 'average_time_on_site_seconds']
).properties(
    title='Average Time on Site Trend by Weekday and Campaign',
    width=700
).interactive()

chart.display()

alt.Chart(...)

**Average time on site by campaign**

In [ ]:
import altair as alt

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('campaign', sort='-y', title='Campaign', axis=alt.Axis(labelAngle=-45)),
    y='average_time_on_site_seconds',
    tooltip=['campaign', 'average_time_on_site_seconds']
).properties(
    title='Average Time on Site by Campaign',
    width=alt.Step(50)
)

chart.display()

alt.Chart(...)

**Average sessions per user by campaign**

In [ ]:
import altair as alt

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('campaign', sort='-y', title='Campaign', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('avg_sessions_per_user', title='Average Sessions Per User'),
    tooltip=['campaign', 'avg_sessions_per_user']
).properties(
    title='Average Sessions Per User by Campaign',
    width=alt.Step(50)
)

chart.display()

alt.Chart(...)

**Total sessions per campaign**

In [ ]:
import altair as alt
import pandas as pd

total_sessions_per_campaign = df.groupby('campaign')['total_sessions'].sum().reset_index()
display(total_sessions_per_campaign)

chart = alt.Chart(total_sessions_per_campaign).mark_bar().encode(
    x=alt.X('campaign', sort='-y', title='Campaign', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('total_sessions', title='Total Sessions'),
    tooltip=['campaign', 'total_sessions']
).properties(
    title='Total Sessions per Campaign',
    width=alt.Step(50)
)

chart.display()

,campaign,total_sessions
0,(data deleted),6825
1,(direct),39528
2,(organic),105828
3,(referral),131452
4,<Other>,34615
5,BlackFriday_V1,9
6,BlackFriday_V2,25
7,Data Share Promo,1469
8,Holiday_V1,19
9,Holiday_V2,37


alt.Chart(...)

**Average time on site by campaign and weekday**

In [ ]:
import altair as alt

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('session_weekday_name', sort=weekday_order, title=None),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    color=alt.Color('session_weekday_name', sort=weekday_order, title='Weekday'),
    column=alt.Column('campaign', header=alt.Header(titleOrient="bottom", labelOrient="bottom", labelAngle=360), title='Campaign')
).properties(
    title='Average Time on Site by Campaign and Weekday',
    width=alt.Step(30)
).interactive()

chart.display()

alt.Chart(...)

**Overall Average time on site by weekday (all campaigns)**

In [ ]:
import altair as alt
import pandas as pd

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

chart = alt.Chart(average_time_on_site_by_weekday).mark_bar().encode(
    x=alt.X('session_weekday_name', sort=weekday_order, title='Weekday'),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    color=alt.Color('session_weekday_name', sort=weekday_order, title='Weekday'),
    tooltip=['session_weekday_name', 'average_time_on_site_seconds']
).properties(
    title='Overall Average Time on Site by Weekday (All Campaigns)',
    width=500
)

chart.display()

alt.Chart(...)

**Overall average time on site by weekday for the high traffic campaings**

In [ ]:
import pandas as pd
import altair as alt

selected_campaigns = ['(data deleted)', '(direct)', '(organic)', '(referral)', '<Other>']
df_selected_campaigns = df[df['campaign'].isin(selected_campaigns)].copy()
average_time_on_site_by_weekday_selected = df_selected_campaigns.groupby('session_weekday_name')['average_time_on_site_seconds'].mean().reset_index()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
average_time_on_site_by_weekday_selected['session_weekday_name'] = pd.Categorical(average_time_on_site_by_weekday_selected['session_weekday_name'], categories=weekday_order, ordered=True)
average_time_on_site_by_weekday_selected = average_time_on_site_by_weekday_selected.sort_values('session_weekday_name')

display(average_time_on_site_by_weekday_selected)


best_day_selected = average_time_on_site_by_weekday_selected.loc[average_time_on_site_by_weekday_selected['average_time_on_site_seconds'].idxmax()]
worst_day_selected = average_time_on_site_by_weekday_selected.loc[average_time_on_site_by_weekday_selected['average_time_on_site_seconds'].idxmin()]

print(f"\nAcross the selected campaigns ({', '.join(selected_campaigns)}):")
print(f"Best day for average time on site: {best_day_selected['session_weekday_name']} ({best_day_selected['average_time_on_site_seconds']:.2f} seconds)")
print(f"Worst day for average time on site: {worst_day_selected['session_weekday_name']} ({worst_day_selected['average_time_on_site_seconds']:.2f} seconds)")

chart = alt.Chart(average_time_on_site_by_weekday_selected).mark_bar().encode(
    x=alt.X('session_weekday_name', sort=weekday_order, title='Weekday'),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    color=alt.Color('session_weekday_name', sort=weekday_order, title='Weekday', scale=alt.Scale(scheme='greens')),
    tooltip=['session_weekday_name', 'average_time_on_site_seconds']
).properties(
    title='Overall Average Time on Site by Weekday (Selected High-Traffic Campaigns)',
    width=500
)

chart.display()

,session_weekday_name,average_time_on_site_seconds
1,Monday,140.580137
5,Tuesday,147.513574
6,Wednesday,156.055021
4,Thursday,159.300179
0,Friday,156.647734
2,Saturday,166.498629
3,Sunday,153.084897



Across the selected campaigns ((data deleted), (direct), (organic), (referral), <Other>):
Best day for average time on site: Saturday (166.50 seconds)
Worst day for average time on site: Monday (140.58 seconds)


alt.Chart(...)

**Average time on site for each weekday across all campaigns**

In [ ]:
# Calculate average time on site for each weekday across all campaigns
average_time_on_site_by_weekday = df.groupby('session_weekday_name')['average_time_on_site_seconds'].mean().reset_index()

# Ensure weekdays are in the correct order for finding min/max and potential visualization
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
average_time_on_site_by_weekday['session_weekday_name'] = pd.Categorical(average_time_on_site_by_weekday['session_weekday_name'], categories=weekday_order, ordered=True)
average_time_on_site_by_weekday = average_time_on_site_by_weekday.sort_values('session_weekday_name')

display(average_time_on_site_by_weekday)

# Identify the best and worst day
best_day = average_time_on_site_by_weekday.loc[average_time_on_site_by_weekday['average_time_on_site_seconds'].idxmax()]
worst_day = average_time_on_site_by_weekday.loc[average_time_on_site_by_weekday['average_time_on_site_seconds'].idxmin()]

print(f"\nAcross all campaigns:")
print(f"Best day for average time on site: {best_day['session_weekday_name']} ({best_day['average_time_on_site_seconds']:.2f} seconds)")
print(f"Worst day for average time on site: {worst_day['session_weekday_name']} ({worst_day['average_time_on_site_seconds']:.2f} seconds)")

,session_weekday_name,average_time_on_site_seconds
1,Monday,94.864562
5,Tuesday,115.312004
6,Wednesday,108.044159
4,Thursday,110.944017
0,Friday,105.222300
2,Saturday,95.928244
3,Sunday,108.729895



Across all campaigns:
Best day for average time on site: Tuesday (115.31 seconds)
Worst day for average time on site: Monday (94.86 seconds)


**EDA**

**Campaign Effectiveness** - The "Total Sessions per Campaign" chart shows which campaigns are driving the most traffic to the site. Campaigns like **'referral', 'organic', and 'direct'** appear to be the largest sources of sessions. While **'other'** also contributes significantly, it's worth investigating what falls into this category. The specific marketing campaigns (e.g., **'BlackFriday', 'Holiday', 'NewYear'**) have significantly fewer total sessions in comparison, and the reason for this is the shorter duration of those campaings.

**Engagement Level** (Average Time on Site) - The "Average Time on Site by Campaign" chart shows which campaigns are keeping users engaged longer on average. We can compare the average time on site for the high-traffic campaigns versus the specific marketing campaigns. Users from campaigns as 'Refereal' and 'Data deleted' tend to spend more time on the site when they visit, and the differencies between the weekdays are not significant. But when we look at the specific campaigns as 'Black Friday', 'Holiday', 'New year' - the differencies in weekdays is high.

**User Loyalty/Return Visits** (Average Sessions Per User) - The "Average Sessions Per User by Campaign" chart indicates how often users from a particular campaign return to the site. A value close to 1 suggests most users only visit once from that campaign during the period, while higher values indicate more repeat visits.

**Weekday Trends within Campaigns** - The "Average Time on Site by Campaign and Weekday" grouped bar chart is interesting for understanding dynamic weekday duration.

**Comparing Weekday Patterns Across Campaigns** - It allows to compare weekday trends between campaigns. The high-traffic campaigns show overall the pick performance on Saturday, but not really significant. And the lowest users engagement is on Monday.The weekday patterns of the specific marketing campaigns are showing the different results. For instance, the peak engagement day for Black Friday_V1 is on Sunday and 'Black Friday_V2' is on Wednesday.

**Across all campaigns we see that the best day for average time on site: Tuesday (115.31 seconds) and the worst day for average time on site: Monday (94.86 seconds)**

**Drawbacks of this Analysis:**

Session Definition: The chosen 30-minute session timeout is an assumption. User behavior can vary, and a different timeout might lead to different session durations.
Lack of Granular Events: The analysis relies on aggregated session duration. It doesn't tell us what users were doing during their sessions or which specific actions contributed to longer/shorter durations.
Attribution Complexity: Users might be exposed to multiple campaigns. This analysis attributes a session to the campaign associated with the event, which might not fully capture the user's journey or the influence of other campaigns.
Data Period: The analysis is based on a specific three-month period. Trends might differ in other times of the year.
Limited User Demographics/Behavior: We don't have information about user demographics or specific browsing behavior within sessions, which could provide deeper insights into why certain campaigns or weekdays have higher engagement.

Recommended Further Analysis:

Sensitivity Analysis on Session Timeout: Test different session timeout values to see how the average time on site and session counts change.
Event-Level Analysis within Sessions: Analyze the sequence and type of events within sessions for different campaigns and weekdays to understand what users are doing.
User Journey Analysis: Track user paths and interactions across multiple sessions and campaigns to understand attribution and cross-campaign influence.
Conversion Rate Analysis: Incorporate conversion metrics (e.g., add-to-cart rate, purchase rate) by campaign and weekday to see if higher engagement on certain days or from certain campaigns translates to business outcomes.
Cohort Analysis: Analyze user behavior over time based on their acquisition campaign to understand long-term engagement and retention.
Segmentation: Analyze these metrics for different user segments (e.g., new vs. returning users, high-value vs. low-value users) to tailor campaign strategies.




**Box plot of Average Time on Site by Campaign and by the weekday**

In [ ]:
import altair as alt

chart_campaign_distribution = alt.Chart(df).mark_boxplot().encode(
    x=alt.X('campaign', sort='-y', title='Campaign', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    tooltip=['campaign', 'average_time_on_site_seconds']
).properties(
    title='Distribution of Average Time on Site by Campaign'
).interactive()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
chart_weekday_distribution = alt.Chart(df).mark_boxplot().encode(
    x=alt.X('session_weekday_name', sort=weekday_order, title='Weekday'),
    y=alt.Y('average_time_on_site_seconds', title='Average Time on Site (seconds)'),
    color=alt.Color('session_weekday_name', sort=weekday_order, title='Weekday'), # Color by weekday
    tooltip=['session_weekday_name', 'average_time_on_site_seconds']
).properties(
    title='Distribution of Average Time on Site by Weekday'
).interactive()

chart_campaign_distribution.display()
chart_weekday_distribution.display()

alt.Chart(...)

alt.Chart(...)